# 03. PromptTemplate과 Message

PromptTemplate을 사용하면 프롬프트를 재사용 가능한 템플릿으로 관리할 수 있습니다.

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

True

## 1. PromptTemplate - 단일 문자열 템플릿

In [2]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("{topic}에 대해 한 문장으로 설명해줘")

# 변수 채우기
prompt = template.invoke({"topic": "RAG"})
print(prompt)

text='RAG에 대해 한 문장으로 설명해줘'


## 2. ChatPromptTemplate - 역할 기반 템플릿

시스템/사용자/AI 메시지를 조합하여 템플릿을 만듭니다.

In [3]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 전문가입니다."),
    ("human", "{question}")
])

messages = chat_prompt.invoke({
    "role": "파이썬",
    "question": "데코레이터가 뭔가요?"
})

print(messages)

messages=[SystemMessage(content='당신은 파이썬 전문가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='데코레이터가 뭔가요?', additional_kwargs={}, response_metadata={})]


## 3. LLM과 연결하여 호출

In [4]:
import sys
sys.path.append("..")
from common_config import llm_connect
model ="gpt-5.4"

temperature = 0
max_tokens = 2086

llm = llm_connect(model=model)
# llm = llm_connect(model=model, temperature=temperature, max_tokens=max_tokens)

messages = chat_prompt.invoke({
    "role": "머신러닝",
    "question": "과적합이 뭔가요?"
})

response = llm.invoke(messages)
print(response.content)

과적합(Overfitting)은 **모델이 학습 데이터에 너무 지나치게 맞춰져서**, 새로운 데이터에서는 성능이 떨어지는 현상입니다.

### 쉽게 말하면
시험 공부를 할 때 **개념을 이해한 게 아니라 기출문제 답만 외운 상태**와 비슷합니다.

- **학습 데이터에서는 점수가 아주 높음**
- **처음 보는 데이터에서는 점수가 낮음**

즉, 데이터의 **진짜 패턴**이 아니라 **잡음(noise)** 이나 우연한 특징까지 외워버린 상태입니다.

---

### 예시
예를 들어 집값을 예측하는 모델을 만든다고 해봅시다.

- 좋은 모델:  
  “집 크기가 크고, 역과 가까울수록 집값이 높아지는 경향이 있네”

- 과적합된 모델:  
  “3번째 학습 데이터의 집은 창문이 5개였으니 가격이 이렇다”처럼  
  학습 데이터의 세세한 우연까지 따라감

---

### 과적합이 생기는 이유
보통 이런 경우에 잘 생깁니다:

- 모델이 너무 복잡할 때
- 데이터가 너무 적을 때
- 학습을 너무 오래 할 때
- 데이터에 잡음이 많을 때

---

### 과적합의 신호
보통 다음처럼 나타납니다:

- **훈련 성능은 매우 좋음**
- **검증/테스트 성능은 나쁨**

예:
- 훈련 정확도: 99%
- 테스트 정확도: 75%

---

### 과적합을 줄이는 방법
대표적으로는:

- **데이터를 더 많이 모으기**
- **모델을 단순하게 만들기**
- **정규화(Regularization) 사용**
- **드롭아웃(Dropout) 사용** (신경망)
- **조기 종료(Early stopping)**
- **교차 검증(Cross validation)**

---

### 반대 개념
과적합의 반대는 **과소적합(Underfitting)** 입니다.

- 과소적합: 너무 단순해서 학습도 잘 못함
- 과적합: 학습은 너무 잘하지만 일반화가 안 됨

---

원하시면 제가 **과적합을 그래프처럼


## 4. 대화 히스토리 포함 - MessagesPlaceholder

이전 대화 내용을 프롬프트에 삽입할 때 사용합니다.

In [5]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

# 이전 대화 기록
history = [
    HumanMessage(content="내 이름은 홍길동이야"),
    AIMessage(content="안녕하세요, 홍길동님!")
]

messages = chat_prompt.invoke({
    "history": history,
    "question": "내 이름이 뭐라고?"
})

In [6]:
messages

ChatPromptValue(messages=[SystemMessage(content='당신은 친절한 AI 어시스턴트입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='내 이름은 홍길동이야', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 홍길동님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={})])

In [7]:
response = llm.invoke(messages)
print(response.content)

홍길동님이세요.


In [8]:
# 현재 대화를 history에 추가
history.append(HumanMessage(content="내 이름이 뭐라고?"))
history.append(AIMessage(content=response.content))
history

[HumanMessage(content='내 이름은 홍길동이야', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 홍길동님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='홍길동님이세요.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

## 5. 다중 변수 템플릿 예제

In [9]:
template = ChatPromptTemplate.from_messages([
    ("system", "당신은 {language} 번역가입니다."),
    ("human", "다음 문장을 번역해줘: {sentence}")
])

messages = template.invoke({
    "language": "영어",
    "sentence": "오늘 날씨가 정말 좋네요."
})

response = llm.invoke(messages)
print(response.content)

The weather is really nice today.


## 문제 해결

In [10]:
template = ChatPromptTemplate.from_messages([
    ("system", """당신은 {sorted_language}에서 {target_language}로 번역하는 번역가입니다.
      출력을 할 때 {sorted_language}로 번역된 문장과 {target_language}로 번역된 문장을 모두 보여주세요.
     
      예시:
      입력(한국어): "오늘 날씨가 정말 좋네요.
      출력(영어): The weather is really nice today. 
     """),
    ("human", "다음 문장을 번역해줘: {text}")
])

messages = template.invoke({
    "sorted_language": "한국어",
    "target_language": "영어",
    "text": "좋은 오후 입니다.."
})

response = llm.invoke(messages)
print(response.content)

한국어: 좋은 오후 입니다..
영어: Good afternoon.
